![RAG Fusion](archs/RAG_fusion.png)

In [74]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.load import loads, dumps

In [8]:
loader = DirectoryLoader(
    path='papers',
    glob='*.pdf',
    loader_cls=PyPDFLoader,
    show_progress=True
)

docs = loader.load()

100%|██████████| 3/3 [00:00<00:00,  3.14it/s]


In [28]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

chunks = text_splitter.split_documents(docs)

In [33]:
# BGE models recommend normalized embeddings for optimal cosine similarity retrieval
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model='BAAI/bge-small-en-v1.5', encode_kwargs={"normalize_embeddings": True}, show_progress=True),
    collection_name="papers",
    persist_directory="./chroma_db",
    collection_metadata={"hnsw:space": "cosine"}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\RadhaKrishna\Documents\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RadhaKrishna\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [64]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "lambda_mult": 0.5})

In [48]:
# Parse the 4 queries into a list
def parse_queries(text):
    return [q.strip() for q in text.strip().split("\n") if q.strip()]

In [91]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.4,
)

In [52]:
multi_query_prompt = PromptTemplate.from_template("""
You are a retrieval expert helping a RAG system.

Generate 4 search queries that would help retrieve complementary information relevant to the user's question.

Requirements:
- Each query should approach the topic from a different angle.
- Use related concepts, technical terms, and alternate phrasings.
- Queries should maximize recall while remaining relevant.
- Avoid generating near-duplicates.
- Do not number the queries.
- Do not provide explanations.
- Output only the queries, one per line.

Question:
{question}
""")

In [92]:
generate_queries_chain = multi_query_prompt | llm | StrOutputParser() | RunnableLambda(parse_queries)

In [97]:
generate_queries_chain.invoke("What is attention mechanism in transformers?")

['scaled dot-product attention mathematical formula query key value matrices',
 'why self-attention replaced recurrent neural networks in sequence modeling',
 'multi-head attention mechanism architecture and benefits in transformers',
 'how attention weights calculate contextual word embeddings in transformer layers']

In [ ]:
# RRF function - This ranks the ranked documents again, no reranking
def reciprocal_rank_fusion(results: list[list], k: int = 60):
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs, start=1):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + k)
    
    return [
        loads(doc)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

In [93]:
rag_fusion_chain = generate_queries_chain | RunnableLambda(lambda queries: [retriever.invoke(q) for q in queries]) | RunnableLambda(reciprocal_rank_fusion)

In [69]:
rag_fusion_chain.invoke("What is attention mechanism in transformers?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\RadhaKrishna\AppData\Local\Temp\ipykernel_23416\861063856.py:12: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  loads(doc)


[Document(metadata={'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-French translation, we outperform the previoussingle state-of-the-art with model by 0.7 BLEU, achieving 

In [71]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [82]:
final_prompt = PromptTemplate.from_template("""
Answer the question based only on the context below.
If the context is insufficient, say I don't know.

Context:
{context}

Question: {original_query}
Answer:""")

In [94]:
full_chain = RunnableParallel({
    "context": rag_fusion_chain | RunnableLambda(format_docs),
    "original_query": RunnablePassthrough()
    }) | final_prompt | llm | StrOutputParser()

In [80]:
print(full_chain.invoke("What is attention mechanism in transformers?"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In Transformers, the attention mechanism, specifically multi-headed self-attention, is fundamental. The Transformer is the first transduction model to rely entirely on self-attention to compute representations of its input and output, replacing recurrent layers.

Self-attention, also known as intra-attention, is an attention mechanism that relates different positions of a single sequence to compute a representation of that sequence. Generally, the output of an attention mechanism is computed as a weighted sum of values, where the weight assigned to each value is determined by a compatibility function of the query with the corresponding key.

The Transformer employs a particular attention mechanism called "Scaled Dot-Product Attention." In this, the input consists of queries, keys, and values. Dot products of the queries and keys are computed, and these are scaled by 1/√dk (where dk is the dimension of queries and keys) to counteract large magnitudes that could push the softmax function

In [95]:
print(full_chain.invoke("If I still love her, how would I get her back?"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

I don't know.
